# CSC 2301 · Week 4 — Strings and files: cleaning a lab report

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W4_strings_files_lab_report.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

Data rarely arrive clean. Laboratories send text reports typed by people: hole names written five
different ways, decimal commas, units glued to numbers, "below detection limit" codes.
This week you turn such a report into a clean table that a program can trust.

**What you will practise:** string methods (`strip`, `lower`, `upper`, `replace`, `split`, `startswith`,
`isdigit`) · slicing · f-strings · reading a text file line by line · writing a CSV file.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/lab_report_raw.txt', 'lab_report_raw.txt'), ('tests/w4/q1.py', 'tests/q1.py'), ('tests/w4/q2.py', 'tests/q2.py'), ('tests/w4/q3.py', 'tests/q3.py'), ('tests/w4/q4.py', 'tests/q4.py'), ('tests/w4/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. Look at the raw file first

Always open a data file and **read it with your eyes** before writing code.

In [ ]:
with open("lab_report_raw.txt", encoding="utf-8") as f:
    text = f.read()
print(text[:900])

What do you notice? The header lines, the separator `;` with or without spaces, hole names such as
`dh-03`, ` DH03` or `dh 03 `, grades like `1,25 g/t`, `<0.01` or `n.d.`, empty lines and a footer.

## 2. Strings are sequences of characters

In [ ]:
s = "  dh-03 "
print(repr(s.strip()))          # remove spaces at both ends
print(s.strip().upper())        # DH-03
print("1,25".replace(",", "."))  # decimal comma -> point
print("S0012;DH-03;0.0-1.5;0.42;".split(";"))
print("DH-03"[3:])              # slicing: from position 3 to the end -> '03'
print("7".zfill(2), "abc12".isdigit(), "12".isdigit())

Strings never change in place: `s.upper()` gives a **new** string, so store it: `s = s.upper()`.

## 3. f-strings for clean output

In [ ]:
hole, n, mean = "DH-05", 11, 0.98765
print(f"{hole}: {n} samples, mean {mean:.2f} g/t")
print(f"{'hole':<8}{'mean':>8}")      # align left / right in 8 characters
print(f"{hole:<8}{mean:>8.3f}")

## 4. Reading a file line by line

A `for` loop over an open file gives one line at a time, **with** its end-of-line character `\n`.

In [ ]:
with open("lab_report_raw.txt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(i, repr(line))
        if i == 6:
            break

---
## Exercises

### Q1. One name per hole

Write `clean_hole_id(s)` that turns any spelling of a hole name into the standard form `DH-` followed by
two digits: `'dh-03'`, `' DH03'`, `'dh 03 '` all become `'DH-03'`, and `'DH-7'` becomes `'DH-07'`.

*Hint:* keep only the characters that are digits.

In [ ]:
def clean_hole_id(s):
    ...

[clean_hole_id(s) for s in ("DH-03", "dh-03", " DH03", "dh 03 ", "DH-7")]

In [ ]:
grader.check("q1")

### Q2. A grade you can compute with

Write `parse_grade(s)` that returns the grade as a float in g/t:

* `'1.25'`, `'1,25'` and `' 0,80 g/t '` → 1.25, 1.25, 0.8
* `'<0.01'` (below the detection limit) → half the limit, **0.005**, a common convention
* `'n.d.'` or an empty string → `None` (unknown is not zero!)

In [ ]:
def parse_grade(s):
    ...

[parse_grade(s) for s in ("1.25", "1,25", " 0,80 g/t ", "<0.01", "n.d.", "")]

In [ ]:
grader.check("q2")

### Q3. Count the samples

Read the file line by line. A **sample line** starts with `S0` and has 5 fields separated by `;`
(after stripping each field). Store in `n_samples` the number of sample lines and in `n_missing`
the number whose grade is `None` according to `parse_grade`.

In [ ]:
n_samples = 0
n_missing = 0
...
n_samples, n_missing

In [ ]:
grader.check("q3")

### Q4. Build a clean table

Build `samples`: a list with one dictionary per sample line, in file order, with the keys
`hole_id` (clean), `from_m` and `to_m` (floats, from the `from-to` field) and `au_gpt`
(from `parse_grade`, may be `None`).

In [ ]:
samples = []
...
samples[:3]

In [ ]:
grader.check("q4")

### Q5. Write the clean file and a summary line

1. Write `lab_report_clean.csv` with the header `hole_id,from_m,to_m,au_gpt` and one line per sample
   **that has a grade** (leave out the `None` ones). The `csv.DictWriter` from the course is the easiest way.
2. Write `summary_line(hole, n, mean)` that returns for example `'DH-03: 12 samples, mean 1.23 g/t'`
   (mean with 2 decimals).

In [ ]:
import csv
...

def summary_line(hole, n, mean):
    ...

summary_line("DH-03", 12, 1.2345)

In [ ]:
grader.check("q5")

### Going further (not checked)

* Print one `summary_line` per hole, using only the samples with a grade.
* Some lines have the remark `RE-ASSAY`. Count them and print their sample numbers.

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()